# Silver Layer: Product Info CDC (Free Edition)

**Pattern**: CDC via `foreachBatch` Merge  
**Trigger**: AvailableNow (Incremental Batch)

In [0]:
from pyspark.sql.functions import col, trim
from delta.tables import DeltaTable

bronze_table = "workspace.bronze.crm_prd_info_cdc"
silver_table = "workspace.silver.crm_prd_info_cdc"
checkpoint_path = "/Volumes/workspace/bronze/raw_sources/checkpoints/silver_crm_prd_info"

def upsert_products(batch_df, batch_id):
    transformed_df = batch_df.select(
        col("prd_id").alias("product_id"),
        trim(col("prd_nm")).alias("product_name"),
        col("prd_cost").alias("cost"),
        col("prd_line").alias("product_line"),
        col("prd_start_dt").alias("start_date"),
        col("prd_end_dt").alias("end_date"),
        col("ingestion_timestamp")
    ).filter(col("product_id").isNotNull())

    if not spark.catalog.tableExists(silver_table):
        transformed_df.write.format("delta").saveAsTable(silver_table)
        return

    target_table = DeltaTable.forName(spark, silver_table)
    (target_table.alias("t")
        .merge(transformed_df.alias("s"), "t.product_id = s.product_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

print(f"Starting incremental update for {silver_table}...")
query = (
    spark.readStream
    .table(bronze_table)
    .writeStream
    .foreachBatch(upsert_products)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()
print(f"✓ Silver update complete. Total records in {silver_table}: {spark.table(silver_table).count():,}")